# PyTorch Lightning baseline template

by Andrés Muñoz-Jaramillo

This notebook is meant to act as a template to train and use a simple regression model to define a baseline that can be compared with a DS application.

It focuses on the concept of defining a PyTorch model, a PyTorch Lightning training loop and the definition of performance metrics.

This notebook assumes familiarity with the concepts of datasets and dataloaders contained in the **_0_dataset_dataloader_template.ipynb_**.

## Set your CUDA visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the CUDA visible device you put here is the one assigned to your team and your machine.   

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader

import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks

torch.set_float32_matmul_precision('medium')



## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model,
including the data it uses. We use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application.

In [3]:
# The config is the single source of truth. load_radioburst_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.radioburst.configs import load_radioburst_config

cfg = load_radioburst_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: radio_burst_forecasting


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from Hugging Face, so re-running this is free.


In [4]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# The linear baseline needs no backbone, so skip the 1.8 GB weights.
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")

Loaded scalers for 13 channels.


## Define Downstream (DS) datasets

This child class takes as input all expected HelioFM parameters, plus additional parameters relevant to the downstream application.  Here we focus in particular on the DS index and parameters necessary to combine it with the HelioFM index.

Another important component of creating a dataset class for your DS is normalization.  Here we use a log normalization on X-ray flux that will act as the output target, making log10(X-ray flux) strictly positive and having 66% of its values between 0 and 1.

In this case we will define both a training and a validation dataset using the indices pointed at in the config.

**_Important:  In this notebook we set max_number_of_samples=6 to potentially avoid going through the whole dataset as we explore it.  Keep in mind this for the future in case the dataset seems smaller than you expect_**


In [5]:
from downstream_apps.radioburst.datasets.radioburst_dataset import RadioBurstDSDataset

In [6]:
# build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling, S3
# access, worker settings) from the config — see notebook 0 for what that block looks
# like written out. Only the radio-burst-specific arguments are passed here, which is
# exactly the list you replace when you fork the template.
#
# It also handles two details that are easy to get wrong by hand: the validation set gets
# phase="val" (no random channel masking or flips), and only the training loader shuffles.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders

max_samples = 170

train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    RadioBurstDSDataset,
    scalers=scalers,
    num_workers=4,          # fewer workers than the script: notebooks start faster
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    max_number_of_samples=max_samples,
    ds_radioburst_folder_path = cfg.data.ds_radioburst_folder_path,
    ds_radioburst_index_file=cfg.data.ds_radioburst_index_file,
    ds_time_column=cfg.data.ds_time_column,
    ds_forecast_horizon=cfg.data.ds_forecast_horizon,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
    ds_spectra_column=cfg.data.ds_spectra_column,
    ds_diagnostics_columns=cfg.data.ds_diagnostics_columns,
    ds_spectra_template_file=cfg.data.ds_spectra_template_file,
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")


train: 116 samples | val: 7 samples | batch_size: 2


Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


In [7]:
# # Inspect a single batch to confirm shapes before training.
# batch = next(iter(train_data_loader))
# print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})

## Define simple baseline model

Defining a simple baseline is important to understand what value the AI model brings to the problem.  

It is always very good to have a very simple baseline model.  Ideally, one that cannot overfit the data.  This is a very good way of really measuring the value added of complex models.   Classical machine learning excels here:

- Regressions and logistic regressions.
- Climatological averages.
- Persistence.
- Simple transformations.

Simple models avoid excessively optimistic assessments of the capabilities of complex models and for many problems are actually remarkably hard to beat.

In this example we define a simple regression acting on the intensity of each channel.  Note that we invert the normalization to deal with strictly positive quantities.  As with the dataset we will be importing the model from a module so that we can use it within training scripts later on.

In [8]:
from downstream_apps.radioburst.models.simple_baseline import TwoStageBurstModel

We can now test that this model manipulates a batch as expected and returns an estimate of burst probability and peak amplitude.

Note that the simple regression model definition requires knowing the number of channels and timesteps, so here we pull that information from the configuration when initializing the model.

In [9]:
n_input_timestamps = cfg.model.time_embedding.time_dim
n_channels = len(cfg.data.channels)

# TwoStageBurstModel needs the flattened input dimension plus the median burst-spectrogram
# template (precomputed by compute_median_template.py, loaded by the dataset itself into
# train_data_loader.dataset.median_spectra_template).
model = TwoStageBurstModel(2 * n_input_timestamps * n_channels, train_data_loader.dataset.median_spectra_template)

Now we can pass the input stack 'ts' to the model to transform it into our regression output.   Note that since this model has not been trained and was initialized randomly, the output here has no real meaning.  It only acts as a test that our model forward doesn't have dimension problems.

Dimension problems are the dominant source of error in this kind of work.

Note that our output has now the size of our batch.

In [10]:
from downstream_apps.radioburst.models.simple_baseline import destandardize_channels

# batch = next(iter(train_data_loader))

# batch_logspace = destandardize_channels(batch, channel_order=cfg.data.channels, scalers=scalers)
# output = model.forward(batch_logspace)
# output["burst_prob"], output["peak_amp"], output["spectra"].shape

## Define your metrics

Metrics are a very important part of training AI models.   They provide your models with the quantification of error, which in turn shifts the weights towards better performing models.  They also provide a way for you to monitor performance, identify overfitting, and quantify value added. 

We now initialize the metrics class which allows you to control what metrics you want to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones for monitoring performance.  As with other components, this takes the form of a loaded module that can later be used in a training script.

In [11]:
from downstream_apps.radioburst.metrics.radioburst_metrics import RadioBurstMetrics

In [12]:
# Both loss terms carry independent weights; 1.0 each is the plain sum. The same
# reasoning as notebook 2's "Balancing the two terms" applies, with peak_amp as the
# regression term. The weights apply to the *_loss modes only.
BURST_WEIGHT = 1.0
PEAK_AMP_WEIGHT = 0.5

train_loss_metrics = RadioBurstMetrics(
    "train_loss", burst_weight=BURST_WEIGHT, peak_amp_weight=PEAK_AMP_WEIGHT
)
val_loss_metrics = RadioBurstMetrics(
    "val_loss", burst_weight=BURST_WEIGHT, peak_amp_weight=PEAK_AMP_WEIGHT
)
train_evaluation_metrics = RadioBurstMetrics("train_metrics")
validation_evaluation_metrics = RadioBurstMetrics("val_metrics")

Now they can be evaluated in our model's output and our ground truth.   First the loss that actually will backpropagate, in this case Mean Squared Error.

In [13]:
# target = {"burst": batch["burst"], "diagnostics": batch["diagnostics"]}
# train_loss_metrics(output, target)

Then a training evaluation that will not backpropagate and inform our model, but that we can keep an eye on. Note that reporting lots of metrics during training will slow the training process.  I'm including it here as an example, but oftentimes it is better to put the diagnostics only in the validation evaluation metrics.

Here we are calculating the Root Relative Squared Error https://lightning.ai/docs/torchmetrics/stable/regression/rse.html 

A value below one means the prediction is better than predicting the average.  It is unlikely that this metric will be lower than one with a randomly initialized model.

In [14]:
# train_evaluation_metrics(output, target)

In the validation evaluation metrics we report both MSE and RRSE.

In [15]:
# validation_evaluation_metrics(output, target)

## Define your PyTorch Lightning module

In this workshop we will use PyTorch Lightning to train our models.  PyTorch Lightning reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the RadioBurstLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

In [16]:
from downstream_apps.radioburst.lightning_modules.pl_simple_baseline import RadioBurstLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [17]:
L.seed_everything(cfg.seed, workers=True)

Seed set to 42


42

## Initialize Lightning module

Now we properly initialize the Lightning module to enable training, including passing the dictionary of metrics.

In [18]:
from functools import partial

metrics = {
    'train_loss': train_loss_metrics,
    'val_loss': val_loss_metrics,
    'train_metrics': train_evaluation_metrics,
    'val_metrics': validation_evaluation_metrics,
}

# Wire up the inverse transform so RadioBurstLightningModule applies it before every model call.
preprocess_fn = partial(
    destandardize_channels,
    channel_order=cfg.data.channels,
    scalers=scalers,
)

lit_model = RadioBurstLightningModule(
    model, metrics, lr=1e-3, batch_size=batch_size, preprocess_fn=preprocess_fn
)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to log in to WandB.  You should have received an invitation to our project.  In order to log in you must:

- Select option 2 (existing account).   In VS Code the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VS Code.

In [19]:
project_name = cfg.wandb_project
run_name = f"radioburst_baseline_full_samples_bw{BURST_WEIGHT}_pa{PEAK_AMP_WEIGHT}"  # encodes the loss weights so the run list stays legible without opening hparams.yaml

wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)

csv_logger = CSVLogger("runs", name=project_name)


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).

In [20]:
max_epochs = 50

# -------------------------------------------------------------------------
# Trainer
# -------------------------------------------------------------------------
# accumulate_grad_batches: batch_size is memory-capped at 2, and with the train set's
# ~44% burst rate a lone batch of 2 has ~31% odds of containing zero burst rows (the
# regression term is then exactly 0.0, see RadioBurstMetrics docstring). Accumulating 4
# micro-batches before each optimizer step gives an effective batch of 8 with <1% odds
# of an all-quiet update, at no extra memory cost. Tune alongside train_burst_rows.
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    accumulate_grad_batches=4,
    logger=[wandb_logger, csv_logger],
    callbacks=[
        ModelCheckpoint(
            dirpath=cfg.output.ckpt_dir,
            filename="baseline-{epoch:02d}-{val_loss_burst_bce:.4f}",
            monitor="val_loss_burst_bce",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
    enable_model_summary=False,
    enable_progress_bar=True,
)

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Fit the model

Finally we fit the model.  We pass the Lightning module and our dataloaders.

In [ ]:
trainer.fit(lit_model, train_data_loader, val_data_loader)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


wandb: Currently logged in as: christianlao (surya-ws2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/home/jovyan/envs/surya_ws/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/jovyan/surya_workshop/downstream_apps/radioburst/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

## Conclusion

With this we have now integrated our dataset, dataloaders, metrics, and baseline into an end-to-end training loop.  The next step is to substitute the simple model with Surya.